# Ablation 1 — Window Size Sensitivity for Segmentation

**Thesis Table**: Section 5.x — Multi-scale vs single-scale segmentation

Tests window sizes [32, 64, 128, 256] individually and combined (multi-scale max-pool)
on the three Reactor 1 labelled signals.

Key finding from thesis:
- Drift (R1-AT-101-PH): w=32 best (F1=0.916), multi-scale = w=32
- Frozen (R1-AT-103-DO): w=32 best (F1=0.747), larger windows dilute short freeze events
- Spikes (R1-AT-102-COND): all windows F1≈0 — 15-point spike diluted even in w=32

In [ ]:
import sys
sys.path.insert(0, '../..')
from dotenv import load_dotenv
load_dotenv('../../.env')

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.timeseer_client import fetch_series_api
from src.models.chatts_loader import load_model
from src.models.mlp import load_mlp
from src.inference.embeddings import encode_text_query
from src.inference.chronos_server import start_server, get_chronos_embedding_cached, shutdown_server
from src.preprocessing.chunking import downsample
from src.segmentation.pipeline import segment_signal

import os
VSC_SCRATCH = os.environ.get('VSC_SCRATCH', '/scratch/leuven/375/vsc37531')
MLP_PATH    = os.path.join(VSC_SCRATCH, 'ChatTS', 'timeseer_data', 'segmentation_mlp_final.pt')
THRESHOLD   = 0.65
WINDOW_SIZES = [32, 64, 128, 256]

print('Imports OK')

In [ ]:
model, tokenizer, processor = load_model('ChatTS-14B')
def _encode(q): return encode_text_query(q, model=model, tokenizer=tokenizer)

start_server()
mlp = load_mlp(MLP_PATH)

In [ ]:
# Signals and their ground-truth anomaly index ranges (in original 8640-pt space)
TEST_SIGNALS = [
    ('R1-AT-101-PH',   'Reactor 1', 'drift',  'Find gradual baseline drift',        0,    8640),
    ('R1-AT-103-DO',   'Reactor 1', 'stale',  'Detect stale data',                  2131, 2259),
    ('R1-AT-102-COND', 'Reactor 1', 'spikes', 'Identify temperature spikes',        1460, 1475),
]

results = {}

for tag, area, anom_type, query, gt_start, gt_end in TEST_SIGNALS:
    print(f'\n── {tag} ({anom_type}) ──')
    vals, idx = fetch_series_api(tag, area)
    vals_ds   = downsample(vals, target=1024)
    n = len(vals_ds)

    # Scale ground-truth indices to downsampled space
    true_s = int(gt_start * n / len(vals))
    true_e = int(gt_end   * n / len(vals))
    true_mask = np.zeros(n)
    true_mask[true_s:true_e] = 1.0

    row = {'tag': tag, 'anom_type': anom_type}
    all_prob_maps = []

    for ws in WINDOW_SIZES:
        prob_map, mask = segment_signal(
            vals_ds, query, mlp, _encode, get_chronos_embedding_cached,
            window_size=ws, threshold=THRESHOLD
        )
        all_prob_maps.append(prob_map)

        # F1 at timestep level
        binary = (prob_map > THRESHOLD).astype(float)
        tp = ((binary == 1) & (true_mask == 1)).sum()
        fp = ((binary == 1) & (true_mask == 0)).sum()
        fn = ((binary == 0) & (true_mask == 1)).sum()
        prec = tp / (tp + fp + 1e-8)
        rec  = tp / (tp + fn + 1e-8)
        f1   = 2 * prec * rec / (prec + rec + 1e-8)
        row[f'w{ws}'] = round(float(f1), 3)
        print(f'  w={ws:3d}: F1={f1:.3f}  TP={int(tp)}  FP={int(fp)}  FN={int(fn)}')

    # Multi-scale: max across all window sizes
    max_prob = np.vstack(all_prob_maps).max(axis=0)
    binary   = (max_prob > THRESHOLD).astype(float)
    tp = ((binary == 1) & (true_mask == 1)).sum()
    fp = ((binary == 1) & (true_mask == 0)).sum()
    fn = ((binary == 0) & (true_mask == 1)).sum()
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    row['multiscale'] = round(float(f1), 3)
    print(f'  Multi-scale: F1={f1:.3f}')

    results[tag] = row

In [ ]:
# Print thesis-ready table
print(f'\n{"Signal":<20} {"GT":<6}', end='')
for ws in WINDOW_SIZES: print(f'  w={ws}', end='')
print('  Multi-scale')
print('-'*75)
for tag, row in results.items():
    gt = {'drift':'A','stale':'C','spikes':'B'}.get(row['anom_type'],'')
    print(f'{tag:<20} {gt:<6}', end='')
    for ws in WINDOW_SIZES: print(f'  {row.get(f"w{ws}",0):.3f}', end='')
    print(f'  {row["multiscale"]:.3f}')

shutdown_server()